# Combined Brand Radar, Social Radar, Culture Converts, and Culture Connects

In [ ]:
import io
import time
import random
import json
import re
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import google.genai as genai
from google.genai import types
from google.cloud import storage, bigquery, secretmanager
from google.api_core import exceptions
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Configuration

In [ ]:
PROJECT_ID = 'converged-brandradar-poc'
DATASET_ID = "brandradar"
REGION = 'europe-west2'
SECRET_ID = "GEMINI_API_KEY"
BUCKET_NAME = 'converged-brandradar'

# Model and Parameters
MODELNAME = 'models/gemini-3.1-pro-preview'
# MODELNAME = 'models/gemini-flash-lite-latest'
# MODELNAME = 'models/gemini-3.0-pro-preview'
# MODELNAME = 'models/gemini-2.5-pro'

# Feature Flags
BRAND = True
SOCIAL = True
CULTURE_CONVERTS = True
CULTURE_CONNECTS = True
TEST = False

REPORTNAME = 'Dominos brand dip'

# Data Sources
DATASOURCE_RADARS = f'gs://{BUCKET_NAME}/lookups/dominos.csv'
DATASOURCE_CONNECTS = f'gs://{BUCKET_NAME}/lookups/csDominos.csv'


# Templates
TEMPLATES = {
    'Brand': f'gs://{BUCKET_NAME}/templates/BrandRadar_Template v2.xlsx',
    'Social': f'gs://{BUCKET_NAME}/templates/SocialRadar_Template v2.xlsx',
    'CultureConverts': f'gs://{BUCKET_NAME}/templates/CultureConverts_template.xlsx',
    'CultureConnects': f'gs://{BUCKET_NAME}/templates/CultureScan_template.xlsx'
}

## Shared Utilities

In [ ]:
def access_secret_version(project_id: str, secret_id: str, version_id: str = "latest") -> str:
    """Accesses the payload for the given secret version."""
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    try:
        response = client.access_secret_version(request={"name": name})
        return response.payload.data.decode("UTF-8")
    except Exception as e:
        print(f"Error accessing secret: {e}")
        return None

def extract_json_robust(response_text):
    """Robust JSON extraction with multiple fallback strategies."""
    if not response_text:
        return None

    for pattern in [r'```json\s*({.*?})\s*```', r'({.*})', r'```json\s*(\[.*?\])\s*```']:
        match = re.search(pattern, response_text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass

    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        return None

def safe_get(data, key, default=None):
    """Safely retrieve a key from a dict with a default fallback."""
    return data.get(key, default) if isinstance(data, dict) else default

def retry_with_backoff(retries=6, backoff_in_seconds=3):
    def decorator(f):
        def wrapper(*args, **kwargs):
            _retries, _backoff = retries, backoff_in_seconds
            attempt = 0
            while _retries > 1:
                attempt += 1
                try:
                    return f(*args, **kwargs)
                except exceptions.ResourceExhausted as e:
                    sleep = _backoff + random.uniform(0, 1)
                    print(f"[{datetime.now().isoformat()}] API quota exceeded (Attempt {attempt}). Retrying ({retries - _retries + 1}/{retries}) in {sleep:.2f}s...")
                    time.sleep(sleep)
                    _backoff *= 2
                    _retries -= 1
                except Exception as e:
                    sleep = _backoff + random.uniform(0, 1)
                    print(f"[{datetime.now().isoformat()}] An error occurred (Attempt {attempt}): {type(e).__name__}: {e}. Retrying ({retries - _retries + 1}/{retries}) in {sleep:.2f}s...")
                    time.sleep(sleep)
                    _backoff *= 2
                    _retries -= 1
            return f(*args, **kwargs)
        return wrapper
    return decorator

@retry_with_backoff(retries=6, backoff_in_seconds=3)
def run_query(client, question):
    call_start_time = time.time()
    print(f"[{datetime.now().isoformat()}] Making API call attempt using model: {MODELNAME}")
    try:
        response = client.models.generate_content(
            model=MODELNAME,
            contents=question,
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="MEDIUM"),
                temperature=1.0,
                tools=[types.Tool(google_search=types.GoogleSearch())]
            )
        )
        call_end_time = time.time()
        duration = call_end_time - call_start_time

        if not response.candidates:
            feedback_str = str(response.prompt_feedback)
            print(f"[{datetime.now().isoformat()}] Request blocked. Feedback: {feedback_str}. Call duration: {duration:.2f}s.")
            return None
        response_text = response.text
        print(f"[{datetime.now().isoformat()}] Successful API response (length: {len(response_text)}). First 100 chars: {response_text[:100]}... Call duration: {duration:.2f}s.")
        return response_text
    except ValueError as e:
        call_end_time = time.time()
        duration = call_end_time - call_start_time
        print(f"[{datetime.now().isoformat()}] API Call Error (ValueError): {e}. Response empty or blocked. Call duration: {duration:.2f}s.")
        return None
    except Exception as e:
        call_end_time = time.time()
        duration = call_end_time - call_start_time
        print(f"[{datetime.now().isoformat()}] API Call Error (General Exception): {type(e).__name__}: {e}. Call duration: {duration:.2f}s.")
        return None

## General Radars Config (Brand, Social, Culture Converts)

In [ ]:
ANALYSIS_CONFIGS = {
    'Brand': {
        'run': BRAND,
        'table_id': 'BrandResults',
        'sheet': 'BrandRadar Results',
        'template': TEMPLATES['Brand'],
        'int_cols': ["mentions", "overallimp", "purchase", "premium", "sustainable",
                     "trust", "recommend", "dynamic", "functional", "personal", "collective"],
        'str_cols': ["touchpoints", "channel", "journey", "branded", "sponsorship", "cgc", "brand_generated"]
    },
    'Social': {
        'run': SOCIAL,
        'table_id': 'SocialResults',
        'sheet': 'SocialRadar Results',
        'template': TEMPLATES['Social'],
        'int_cols': ["sentiment", "emotion", "positivity", "shift", "spike", "themes",
                     "emerging_topics", "debates", "platforms", "engagement", "media_type"],
        'str_cols': []
    },
    'CultureConverts': {
        'run': CULTURE_CONVERTS,
        'table_id': 'CultureConverts',
        'sheet': 'Cultural Converts',
        'template': TEMPLATES['CultureConverts'],
        'int_cols': [f"Buzz{i}" for i in range(1, 6)] + [f"Belong{i}" for i in range(1, 6)] + \
                    [f"Belief{i}" for i in range(1, 6)] + [f"Behave{i}" for i in range(1, 6)],
        'str_cols': [f"Buzz{i}Desc" for i in range(1, 6)] + [f"Belong{i}Desc" for i in range(1, 6)] + \
                    [f"Belief{i}Desc" for i in range(1, 6)] + [f"Behave{i}Desc" for i in range(1, 6)]
    }
}

# Setup Columns Schema
for t, c in ANALYSIS_CONFIGS.items():
    if t == 'CultureConverts':
        c['columns'] = ["Brand", "Category", "Market"] + c['int_cols'] + c['str_cols'] + \
        ["date", "projectname"]
    else:
        c['columns'] = ["Brand", "Category", "Market", "Summary"] + \
                       c['int_cols'] + \
                       [f"{col}_description" for col in c['int_cols']] + \
                       [f"{col}_description" for col in c['str_cols']] + \
                       ["date"]


In [ ]:
def initialize_dataframe(task_name):
    """Dynamically set up DataFrame based on schema."""
    cols = ANALYSIS_CONFIGS[task_name]['columns']
    df = pd.DataFrame(columns=cols)
    # Conditional dtype assignment for int_cols
    for col in ANALYSIS_CONFIGS[task_name]['int_cols']:
        if task_name == 'CultureConverts':
            df[col] = pd.Series(dtype='float64')
        else:
            df[col] = pd.Series(dtype='int64')
    df['date'] = pd.Series(dtype='datetime64[ns]')
    return df

def get_question_radars(task_name, brand, category, market):
    if task_name == 'Brand':
        return f"""Please provide a brief (approximately 200 words) summary of the {brand} brand's perception in the {category} category in the {market} market.

          For each of the following aspects, provide a score out of a 100 with a very short description of the score:
        - How often is it mentioned?
        - What is your overall impression of the brand?
        - Have people purchased the brand?
        - Is it thought of as a premium brand that people would pay a premium price for?
        - It is considered a sustainable brand?
        - do people trust it?
        - Would they recommend it to others?
        - What do you think of it's speed, dynamism and willingness to adapt to change?
        - Does the brand meet functional requirements, for example, seamless digital, in-store or purchasing experience, product range, highly innovative,
        value for money, offers price consistency, time saving, high quality products, respectful of customers and their data, offers exclusive experiences,
        offers unique products and services, safe products and services, acts like a leader or challenger brand, has a good reputation, delivers on what it says?        - Does the brand meet personal requirements, for example, image enhancing, personalised offers, confidence building, health enhancing, lets me escape from the everyday, inspires me with new opportunities and ideas,
        - Does the brand meet personal requirements, for example, image enhancing, personalised offers, confidence building, health enhancing,
        lets me escape from the everyday, inspires me with new opportunities and ideas, instilling peace of mind and confidence, connecting people,
        enables me to be smart with money and time, simplifies my life, helps me express myself as an individual, feel more attractive and stylish,
        make me feel special and unique, makes me energised and alive, helps me feel in control of my life, gives me a sense of happiness, makes me feel good,
        makes me feel like it is looking after my best interests?
         - Does the brand meet collective requirements, for example does it act with integrity, is it ethical, is it transparent in its operations,
         is it considered a good employer, offers sustainable products, it fights against poverty and hunger, supports healthy living and promotes well-being,
         supports culture and education, supports equality, diversity and inclusion, supports social issues and good causes,
         contributes to the national economy and my local community, invests in innovative, sustainable and ethical solutions, inspires sustainable,
         responsible behaviours and consumption, committed to making products/services more sustainable?

        and can you provide short descriptions of the following questions
        - What are the most common touchpoints associated with the brand?
        - Which channel do consumers mention most when discussing the brand?
        - Where has the brand appeared most in the consumer journey online
        - What formats of branded content does brand most often use?
        - What recent sponsorships or partnerships has the brand activated?
        - What kinds of consumer-generated content appear in discussions about the brand?
        - What kinds of brand-generated content appear in discussions about the brand?

        Could the results be returned in JSON format with the brand in a variable called brand, the country in a field called market, the summary in a field called summary,
        the individual scores and the short descriptions of the scores in fields called, mentions, mentions_description, overallimp, overallimp_description, purchase, purchase_description, premium, premium_description,
        sustainable, sustainable_description, trust, trust_description, recommend, recommend_description, dynamic, dynamic_description, functional, functional_description, personal, personal_description, collective, collective_description,
        touchpoints_description, channel_description, journey_description, branded_description, sponsorship_description, cgc_description, brand_generated_description?
  """

    elif task_name == 'Social':
        return f"""Please provide a brief (approximately 200 words) summary of the {brand} brand’s public sentiment and online conversation dynamics in the {category} category in the {market} market based on recent digital signals.

        	For each of the following aspects, provide a score out of 100 and a short description explaining the score:
            - What is the general sentiment about the brand?
            - What emotions are most commonly associated with the brand?
            - Is the brand currently viewed more positively overall?
            - Is the brand currently viewed more negatively overall?
            - Are there noticeable sentiment shifts over the last 6 months?
            - Have there been sentiment spikes triggered by campaigns, news, or events?
            - What are the most discussed themes in online conversations about the brand?
            - What social, cultural, product or service topics are emerging?
            - Are there any polarizing or divisive conversations surrounding the brand?
            - Which platforms (e.g. Reddit, X, TikTok, forums, blogs) host the most conversations about the brand?
            - Where does the brand get the most engagement?
            - Is brand visibility driven more by owned channels or third-party/earned media?

          Please return results in **JSON format** the using the following fields:
          - `"brand"` for the brand name
          - `"market"` for the country
          - `"summary"` for the narrative overview
          - `"scores"` as an object with:
            - `sentiment`, `emotion`, `positivity`, `sentiment_shift`, `sentiment_spike`
            - `themes`, `emerging_topics`, `debates`
            - `platforms`, `engagement`, `media_type`

          Each object should contain:
          - `score` (0–100)
          - `description` (1–2 sentence explanation)"""

    elif task_name == 'CultureConverts':

        return f"""Please analyze the {brand} brand's perception in the {category} category in the {market} market.
            I am interested in understanding 4 areas, buzz, belonging, belief and behaviour. Could provide a value between 1 and 100 for the following questions and store them in JSON format,
            can you provide a brief (approximately 50-100 words) summary of the reasons behind these scores?

            Buzz1: Measuring attention. Number of mentions (total #) - across social, news, forums.
            Buzz2: Share of cultural voice. % of culturally relevant mentions vs peer brands in the target market.
            Buzz3:  Competitor share. Share of voice vs competitors (%) - brand mentions as a % of category mentions.
            Buzz4: Authority-weighted presence. Weighted score based on coverage in high-authority media and top creators.
            Buzz5: Topic spread. Count of distinct cultural themes (e.g., sustainability, arts, finance) where the brand appears organically.
            Belong1: Cultural fit score. % of contextual mentions coded as “authentic” vs “performative”.
            Belong2: Community acceptance index. Proportion of mentions coming from identified target communities/subcultures (%).
            Belong3: Peer comparison on affinity. Relative score vs. category peers for perceived cultural fit.
            Belong4: Audience inclusion signal. Frequency of language indicating inclusion (e.g., “for us,” “represents,” “part of”, “this is for people like”).
            Belong5: Negative fit incidence. % of mentions expressing rejection or misalignment with cultural spaces.
            Belief1: Cultural authority score. % of mentions framing the brand as a leader, innovator, or credible voice in cultural or social topics.
            Belief2: Value alignment index. Frequency of positive associations with values (e.g., sustainability, diversity, ethics) in cultural discourse.
            Belief3: Thought leadership presence. Number of named endorsements or quotes from recognised cultural leaders/experts.
            Belief4: Peer Comparison on Authority. Relative score vs category peers for perceived leadership and credibility.
            Belief5: Negative Authority Incidence. % of mentions questioning or criticizing the brand’s intent or values.
            Behave1: Action conversation signals. Frequency of mentions indicating concrete action (e.g., “I attended”, “I bought”, “I downloaded”, purchase, trial, sign-up).
            Behave2: Advocacy rate. % of user-generated content showing active recommendation or endorsement of the brand.
            Behave3: Cultural participation index. Number of instances where audiences join brand-led cultural initiatives (e.g., events, challenges, collaborations).
            Behave4: Behavioural shift mentions. Mentions where users report changing habits or choices influenced by the brand’s cultural activity.
            Behave5: Peer comparison on adoption. Relative score vs category peers for observable cultural-driven actions

            These should be provided in JSON
            format with fields: Brand, Category, Market, Buzz1, Buzz2, Buzz3, Buzz4, Buzz5, Belong1, Belong2, Belong3,
            Belong4, Belong5, Belief1, Belief2, Belief3, Belief4, Belief5, Behave1, Behave2, Behave3, Behave4, Behave5,
            Buzz1Desc, Buzz2Desc, Buzz3Desc, Buzz4Desc, Buzz5Desc, Belong1Desc, Belong2Desc, Belong3Desc, Belong4Desc,
            Belong5Desc, Belief1Desc, Belief2Desc, Belief3Desc, Belief4Desc, Belief5Desc, Behave1Desc, Behave2Desc,
            Behave3Desc, Behave4Desc,
            Behave5Desc,
            date, projectname

            CRITICAL: Return ONLY a valid JSON object with NO markdown formatting, NO additional text."""
    else:
        raise ValueError(f"Invalid task name: {task_name}")

def extract_values_to_row(task_name, json_data, row_template):
    if not json_data:
        return row_template

    config = ANALYSIS_CONFIGS[task_name]
    new_row = row_template.copy()

    if task_name != 'CultureConverts':
        new_row['Summary'] = json_data.get('summary', '')

    if task_name == 'Social':
        scores = json_data.get('scores', {})
        for measure in config['int_cols']:
            key = "sentiment_shift" if measure == "shift" else "sentiment_spike" if measure == "spike" else measure
            if key in scores:
                new_row[measure] = scores[key].get('score', 0)
                new_row[f"{measure}_description"] = scores[key].get('description', '')
    else:
        for k, v in json_data.items():
            if k in new_row or k not in config['columns']: continue
            new_row[k] = v

    return new_row


## Culture Connects specific config

In [ ]:
DFSUPERSPACE = pd.DataFrame({
    "superspace": [
        "Body & Wellness",
        "Community & Society",
        "Entertainment & Story",
        "Food & Drink",
        "Home & Away",
        "Learning & Creation",
        "Sport & Play",
        "Style & Aesthetics",
    ],
    "activities": [
        "Alternative/Spiritual Wellness,Fitness/Training,Mental Health & Mindfulness,Nutrition/Biohacking,Outdoor Wellness,Recovery/Sleep coaching",
        "Activism,Faith/Spirituality,Local Scenes,Politics,Volunteering",
        "Books & Comics,Film/TV/Streaming,Gaming & Esports,Live Performance (theatre/comedy),Music & Nightlife,Podcasts/Audio",
        "Baking,Coffee/Tea,Dining Out,Home Cooking,Markets & Pop-ups,No/Low Alcohol,Specialty Cuisines",
        "DIY/Maker,Gardening,Household Rituals,Interiors,Pets,Smart Home",
        "Arts & Crafts,Coding/Maker/3D,Languages,Online Learning,Photography/Filmmaking,Writing & Publishing",
        "Dance,Martial Arts,Outdoor/Adventure,Tabletop/Board Games,Team & Individual Sports",
        "Beauty/Grooming,Body Art (tattoos/piercings),Collectibles/Luxury,Design/Architecture,Fashion,Streetwear",
    ],
})

def setscandf():
    df = pd.DataFrame(columns=[
        "Market", "Audience", "SuperSpace",
        "Activity", "InterestLevel", "WeeklyTimeSpenthours", "AnnualSpendDollars",
        "InterestLevelMarket", "WeeklyTimeSpentMarket", "AnnualSpendMarket",
        "Rationale", "Date", "Report",
    ])
    for col in ["InterestLevel", "WeeklyTimeSpenthours", "AnnualSpendDollars",
                "InterestLevelMarket", "WeeklyTimeSpentMarket", "AnnualSpendMarket"]:
        df[col] = df[col].astype("float64")
    return df

def get_question_connects(market, audience, activities):
    return f"""For {audience} in the {market} market
I need an estimation of the interest level (on a scale 1-10), the average monthly time spend (in hours),
the annual spend (in US Dollars), and a short ~100-word description of your rationale for the scores
for the following activities:
  {activities}

Provide ONLY a JSON object with a key "data" containing an array of objects.
Each object must use these exact keys:
  activity_name          (string)
  interest_level         (number 1-10)
  weekly_time_hours      (float)
  annual_spend_usd       (float)
  interest_level_market  (number 1-10)
  weekly_time_hours_market (float)
  annual_spend_usd_market  (float)
  rationale              (string, ~100 words)
"""

def add_to_dataframescan(df, market, audience, superspace, response_text):
    brand_json = extract_json_robust(response_text)
    now = pd.Timestamp.now().normalize()

    def error_row(msg):
        return {
            "Market": market, "Audience": audience, "SuperSpace": superspace,
            "Activity": "Parse Error",
            "InterestLevel": None, "WeeklyTimeSpenthours": None, "AnnualSpendDollars": None,
            "InterestLevelMarket": None, "WeeklyTimeSpentMarket": None, "AnnualSpendMarket": None,
            "Rationale": msg, "Date": now, "Report": REPORTNAME,
        }

    if brand_json is None:
        return pd.concat([df, pd.DataFrame([error_row("Failed to extract JSON")])], ignore_index=True)

    data_list = safe_get(brand_json, "data")
    if not isinstance(data_list, list):
        return pd.concat([df, pd.DataFrame([error_row("JSON 'data' field is not a list")])], ignore_index=True)

    new_rows = []
    for item in data_list:
        new_rows.append({
            "Market":          market,
            "Audience":        audience,
            "SuperSpace":      superspace,
            "Activity":        safe_get(item, "activity_name", ""),
            "InterestLevel":        safe_get(item, "interest_level", 0),
            "WeeklyTimeSpenthours":       safe_get(item, "weekly_time_hours", 0),
            "AnnualSpendDollars":      safe_get(item, "annual_spend_usd", 0),
            "InterestLevelMarket":  safe_get(item, "interest_level_market", 0),
            "WeeklyTimeSpentMarket": safe_get(item, "weekly_time_hours_market", 0),
            "AnnualSpendMarket":safe_get(item, "annual_spend_usd_market", 0),
            "Rationale":     safe_get(item, "rationale", ""),
            "Date":            now,
            "Report":     REPORTNAME,
        })

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    return df

## Multi-Task Export Engine

In [ ]:
def export_results(task_name, df, storage_client, all_responses=None):
    bucket = storage_client.bucket(BUCKET_NAME)

    # 1. JSON
    if all_responses:
        blob_name = f"results/{task_name}_{REPORTNAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        bucket.blob(blob_name).upload_from_string(json.dumps(all_responses, indent=4), content_type='application/json')
        print(f"JSON Exported: gs://{BUCKET_NAME}/{blob_name}")

    # 2. XLS
    if task_name == 'CultureConnects':
        sheet_name = 'Culture Scan'
        xls_path = f"gs://{BUCKET_NAME}/results/CultureScan_{REPORTNAME}_{datetime.now().strftime('%Y%m%d')}.xlsx"
    else:
        config = ANALYSIS_CONFIGS[task_name]
        sheet_name = config['sheet']
        xls_path = f"gs://{BUCKET_NAME}/results/{task_name}_{REPORTNAME.strip()}.xlsx"

    template_url = TEMPLATES[task_name]
    template_blob = bucket.blob('/'.join(template_url.split('/')[3:]))
    template_buffer = io.BytesIO()
    template_blob.download_to_file(template_buffer)
    template_buffer.seek(0)

    book = load_workbook(template_buffer)
    book['Key'].cell(row=3, column=1, value=REPORTNAME)
    sheet = book[sheet_name]

    for r_idx, out_row in enumerate(dataframe_to_rows(df, index=False, header=False), 2):
        for c_idx, value in enumerate(out_row, 1):
            sheet.cell(row=r_idx, column=c_idx, value=value)

    out_buffer = io.BytesIO()
    book.save(out_buffer)
    out_buffer.seek(0)

    out_blob_path = '/'.join(xls_path.split('/')[3:])
    bucket.blob(out_blob_path).upload_from_file(out_buffer, content_type='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
    print(f"Excel Exported: {xls_path}")

    # 3. BigQuery
    bq_client = bigquery.Client(project=PROJECT_ID)
    if task_name == 'CultureConnects':
        table_id = "CultureConnects"
        schema = [
            bigquery.SchemaField("market", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("audience", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("superspace", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("activity", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("InterestLevel", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("WeeklyTimeSpenthours", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("AnnualSpendDollars", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("InterestLevelMarket", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("WeeklyTimeSpentMarket", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("AnnualSpendMarket", "FLOAT", mode="NULLABLE"),
            bigquery.SchemaField("Rationale", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("Date", "DATE", mode="NULLABLE"), # Changed to DATE
            bigquery.SchemaField("Report", "STRING", mode="NULLABLE"),
        ]

        df_bq = df.rename(columns={
            "Market": "market", "Audience": "audience", "SuperSpace": "superspace", "Activity": "activity",
        })
        for col in ["InterestLevel", "WeeklyTimeSpenthours", "AnnualSpendDollars", "InterestLevelMarket", "WeeklyTimeSpentMarket", "AnnualSpendMarket"]:
            if col in df_bq.columns:
                df_bq[col] = pd.to_numeric(df_bq[col], errors="coerce").astype(float)

        if "Date" in df_bq.columns:
            df_bq["Date"] = pd.to_datetime(df_bq["Date"], errors='coerce').dt.date # Convert to datetime.date object

        table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_id}"
        job_config = bigquery.LoadJobConfig(schema=schema, write_disposition=bigquery.WriteDisposition.WRITE_APPEND)

        try:
            bq_client.load_table_from_dataframe(df_bq, table_ref, job_config=job_config).result()
            print(f"BigQuery Updated: {table_id} ({len(df)} rows)")
        except Exception as e:
            print(f"BigQuery Error: {e}")

    else: # This block handles Brand, Social, CultureConverts
        table_id = config['table_id']
        table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_id}"

        df_bq = df.copy()

        # Convert 'date' column to string format if it exists, ensuring final type is string
        if "date" in df_bq.columns:
            df_bq["date"] = df_bq["date"].dt.strftime('%Y-%m-%d').astype(str) # Explicitly cast to string

        schema = []
        for col in df_bq.columns:
            dtype = df_bq[col].dtype
            if col == "date":
                bq_type = "STRING"
            elif dtype == 'int64':
                bq_type = "INTEGER"
            elif dtype == 'float64':
                bq_type = "FLOAT"
            else:
                bq_type = "STRING"
            schema.append(bigquery.SchemaField(col, bq_type))

        # Always use WRITE_APPEND as per instruction, providing the schema
        job_config = bigquery.LoadJobConfig(schema=schema, write_disposition=bigquery.WriteDisposition.WRITE_APPEND)

        try:
            bq_client.load_table_from_dataframe(df_bq, table_ref, job_config=job_config).result()
            print(f"BigQuery Updated: {table_id} ({len(df)} rows)")
        except Exception as e:
            print(f"BigQuery Error: {e}")


## Execution Engines

In [ ]:
def execute_radars_task(task_name, df_input, client, storage_client):
    import urllib.request
    print(f"\n{'='*40}\nProcessing Task: {task_name} (BATCH)\n{'='*40}")

    df = initialize_dataframe(task_name)
    all_responses = []

    # 1. Prepare Batch Requests
    requests = []
    for idx, row in df_input.iterrows():
        question = get_question_radars(task_name, row['brand'], row['category'], row['market'])
        requests.append({
            'custom_id': str(idx),
            'contents': [{'role': 'user', 'parts': [{'text': question}]}]
        })

    print(f"Submitting {len(requests)} requests to Batch API...")
    batch_job = client.batches.create(
        model=MODELNAME,
        requests=requests
    )
    print(f"Batch job {batch_job.name} started.")

    # 2. Wait for Completion
    while True:
        job = client.batches.get(name=batch_job.name)
        if job.state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
            break
        print(f"[{datetime.now().isoformat()}] Job state: {job.state}, sleeping 30s...")
        time.sleep(30)

    if job.state != 'SUCCEEDED':
        print(f"Batch job failed or cancelled. Final state: {job.state}")
        return

    print("Job succeeded! Fetching and parsing results...")

    # 3. Fetch Output
    req = urllib.request.Request(job.output_uri)
    try:
        with urllib.request.urlopen(req) as response:
            lines = response.read().decode('utf-8').strip().split('\n')
    except Exception as e:
        print(f"Failed to fetch output_uri directly: {e}, falling back to auth...")
        req.add_header('Authorization', f'Bearer {access_secret_version(PROJECT_ID, SECRET_ID)}')
        with urllib.request.urlopen(req) as response:
             lines = response.read().decode('utf-8').strip().split('\n')

    # map outputs by custom_id
    results_map = {}
    for line in lines:
        if not line.strip(): continue
        res = json.loads(line)
        custom_id = int(res.get('custom_id', -1))
        
        resp_text = None
        try:
            if 'response' in res and 'candidates' in res['response'] and len(res['response']['candidates']) > 0:
                resp_text = res['response']['candidates'][0]['content']['parts'][0]['text']
        except KeyError:
            pass
            
        results_map[custom_id] = resp_text

    # 4. Build DataFrame
    for idx, row in df_input.iterrows():
        resp_text = results_map.get(idx)
        question = get_question_radars(task_name, row['brand'], row['category'], row['market'])

        brand_json = extract_json_robust(resp_text) if resp_text else None

        row_template = {
            "Brand": brand_json.get('brand', row['brand']) if brand_json else row['brand'],
            "Category": row['category'],
            "Market": row['market'],
            "date": pd.Timestamp.now().normalize()
        }
        if task_name == 'CultureConverts':
            row_template["projectname"] = REPORTNAME

        row_data = extract_values_to_row(task_name, brand_json, row_template)
        df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)

        all_responses.append({
            "timestamp": datetime.now().isoformat(),
            "brand": row['brand'],
            "market": row['market'],
            "input_question": question,
            "gemini_response": resp_text,
            "success": bool(brand_json)
        })

    df['projectname'] = REPORTNAME

    if task_name == 'CultureConverts':
        print("Adding calculated fields for CultureConverts task...")
        with pd.option_context("future.no_silent_downcasting", True):
            df['Buzzscore'] = (df['Buzz1'] + df['Buzz2'] + df['Buzz3'] + df['Buzz4'] + df['Buzz5']) / 5
            df['Belongscore'] = (df['Belong1'] + df['Belong2'] + df['Belong3'] + df['Belong4'] + (100 - df['Belong5'])) / 5
            df['Beliefscore'] = (df['Belief1'] + df['Belief2'] + df['Belief3'] + df['Belief4'] + (100 - df['Belief5'])) / 5
            df['Behavescore'] = (df['Behave1'] + df['Behave2'] + df['Behave3'] + df['Behave4'] + df['Behave5']) / 5
            df['CultureScore'] = (df['Buzzscore'] + df['Belongscore'] + df['Beliefscore'] + df['Behavescore']) / 4

    export_results(task_name, df, storage_client, all_responses)

def execute_connects_task(df_input, client, storage_client):
    import urllib.request
    print(f"\n{'='*40}\nProcessing Task: CultureConnects (BATCH)\n{'='*40}")

    df = setscandf()
    all_responses = []

    # 1. Prepare Batch Requests
    requests = []
    req_meta = {}
    req_idx = 0
    for idx, row in df_input.iterrows():
        for _, rowss in DFSUPERSPACE.iterrows():
            superspace = rowss["superspace"]
            question = get_question_connects(row["market"], row["audience"], rowss["activities"])
            
            requests.append({
                'custom_id': str(req_idx),
                'contents': [{'role': 'user', 'parts': [{'text': question}]}]
            })
            req_meta[str(req_idx)] = {
                'market': row["market"],
                'audience': row["audience"],
                'superspace': superspace,
                'question': question
            }
            req_idx += 1

    print(f"Submitting {len(requests)} requests to Batch API for CultureConnects...")
    batch_job = client.batches.create(
        model=MODELNAME,
        requests=requests
    )
    print(f"Batch job {batch_job.name} started.")

    # 2. Wait for Completion
    while True:
        job = client.batches.get(name=batch_job.name)
        if job.state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
            break
        print(f"[{datetime.now().isoformat()}] Job state: {job.state}, sleeping 30s...")
        time.sleep(30)

    if job.state != 'SUCCEEDED':
        print(f"Batch job failed or cancelled. Final state: {job.state}")
        return

    print("Job succeeded! Fetching and parsing results...")

    # 3. Fetch Output
    req = urllib.request.Request(job.output_uri)
    try:
        with urllib.request.urlopen(req) as response:
            lines = response.read().decode('utf-8').strip().split('\n')
    except Exception as e:
        print(f"Failed to fetch output_uri directly: {e}")
        req.add_header('Authorization', f'Bearer {access_secret_version(PROJECT_ID, SECRET_ID)}')
        with urllib.request.urlopen(req) as response:
             lines = response.read().decode('utf-8').strip().split('\n')

    # map outputs by custom_id
    results_map = {}
    for line in lines:
        if not line.strip(): continue
        res = json.loads(line)
        custom_id = str(res.get('custom_id', ''))
        
        resp_text = None
        try:
            if 'response' in res and 'candidates' in res['response'] and len(res['response']['candidates']) > 0:
                resp_text = res['response']['candidates'][0]['content']['parts'][0]['text']
        except KeyError:
            pass
            
        results_map[custom_id] = resp_text

    # 4. Build DataFrame
    for custom_id, meta in req_meta.items():
        resp_text = results_map.get(custom_id)
        
        df = add_to_dataframescan(df, meta['market'], meta['audience'], meta['superspace'], resp_text)

        all_responses.append({
            "timestamp":       datetime.now().isoformat(),
            "market":          meta['market'],
            "audience":        meta['audience'],
            "superspace":      meta['superspace'],
            "input_question":  meta['question'],
            "gemini_response": resp_text,
            "success":         extract_json_robust(resp_text) is not None,
        })

    export_results('CultureConnects', df, storage_client, all_responses)


## Entrypoint

In [ ]:
def main():
    start_time = time.time()

    api_key = access_secret_version(PROJECT_ID, SECRET_ID)
    if not api_key:
        print("Failed to retrieve API key!")
        return

    client = genai.Client(api_key=api_key)
    storage_client = storage.Client()

    # 1. Run Core Radars (if any are active)
    if any([BRAND, SOCIAL, CULTURE_CONVERTS]):
        dfbrands_radars = pd.read_csv(DATASOURCE_RADARS)
        if TEST:
            dfbrands_radars = dfbrands_radars.head(1)
        cats = dfbrands_radars.groupby(['market', 'category']).size().reset_index()
        cats['brand'] = 'All Category'
        dfbrands_radars = pd.concat([dfbrands_radars, cats], ignore_index=True)

        for task_name, config in ANALYSIS_CONFIGS.items():
            if config['run']:
                df_to_run_radars = dfbrands_radars.copy()
                execute_radars_task(task_name, df_to_run_radars, client, storage_client)

    # 2. Run Culture Connects (if active)
    if CULTURE_CONNECTS:
        dfbrands_connects = pd.read_csv(DATASOURCE_CONNECTS)
        if TEST:
            dfbrands_connects = dfbrands_connects.head(1)
        execute_connects_task(dfbrands_connects, client, storage_client)

    print(f"\nTotal Run time: {time.time() - start_time:.2f} seconds")

if __name__ == "__main__":
    main()



Processing Task: Brand
[1/13] Processing Domino's (UK)
[2026-05-07T16:16:02.808724] Making API call attempt using model: models/gemini-3.1-pro-preview
[2026-05-07T16:16:52.634399] Successful API response (length: 4549). First 100 chars: ```json
{
  "brand": "Domino's",
  "market": "UK",
  "summary": "Domino's is a powerhouse in the UK ... Call duration: 49.83s.
[2/13] Processing mcdonald's (UK)
[2026-05-07T16:16:53.638336] Making API call attempt using model: models/gemini-3.1-pro-preview
[2026-05-07T16:17:37.075871] Successful API response (length: 4400). First 100 chars: ```json
{
  "brand": "McDonald's",
  "market": "UK",
  "summary": "McDonald's holds a dominant and u... Call duration: 43.44s.
[3/13] Processing kfc (UK)
[2026-05-07T16:17:38.078841] Making API call attempt using model: models/gemini-3.1-pro-preview
[2026-05-07T16:18:27.968245] Successful API response (length: 5353). First 100 chars: ```json
{
  "brand": "KFC",
  "market": "UK",
  "summary": "KFC holds a prominent a